In [4]:
# Install dependencies
!pip install fastapi uvicorn pyngrok sqlalchemy pydantic nest_asyncio

# Imports
from fastapi import FastAPI, Depends, HTTPException
from sqlalchemy import create_engine, Column, Integer, String, Float, Date
from sqlalchemy.orm import sessionmaker, declarative_base, Session
from pydantic import BaseModel
from datetime import date
from pyngrok import ngrok
import uvicorn, threading, nest_asyncio

nest_asyncio.apply()

# 🔑 ADD YOUR NGROK TOKEN HERE
ngrok.set_auth_token("33mt2xxmAFi5hWhw721sNJNvxGj_2ZbUYhpE9xzHbKJN6XmZL")

# Database setup
DATABASE_URL = "sqlite:///./finance.db"
engine = create_engine(DATABASE_URL, connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(bind=engine)
Base = declarative_base()

# Models
class Transaction(Base):
    __tablename__ = "transactions"
    id = Column(Integer, primary_key=True)
    amount = Column(Float)
    type = Column(String)
    category = Column(String)
    date = Column(Date)
    notes = Column(String)

class User(Base):
    __tablename__ = "users"
    id = Column(Integer, primary_key=True)
    username = Column(String, unique=True)
    password = Column(String)
    role = Column(String)

Base.metadata.create_all(bind=engine)

# Schemas
class TransactionCreate(BaseModel):
    amount: float
    type: str
    category: str
    date: date
    notes: str

class UserCreate(BaseModel):
    username: str
    password: str
    role: str

# App
app = FastAPI()

# DB Dependency
def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()

# Role check
def check_role(user_role, allowed_roles):
    if user_role not in allowed_roles:
        raise HTTPException(status_code=403, detail="Not authorized")

# Home
@app.get("/")
def home():
    return {"message": "Finance API Running 🚀"}

# ---------------- USER APIs ----------------
@app.post("/users/")
def create_user(user: UserCreate, db: Session = Depends(get_db)):
    db_user = User(**user.dict())
    db.add(db_user)
    db.commit()
    db.refresh(db_user)
    return db_user

@app.post("/login/")
def login(username: str, password: str, db: Session = Depends(get_db)):
    user = db.query(User).filter(User.username == username).first()
    if not user or user.password != password:
        raise HTTPException(status_code=401, detail="Invalid credentials")
    return {"message": "Login successful", "role": user.role}

# ---------------- TRANSACTION APIs ----------------
@app.post("/transactions/")
def add_transaction(transaction: TransactionCreate, role: str, db: Session = Depends(get_db)):
    check_role(role, ["admin"])
    db_transaction = Transaction(**transaction.dict())
    db.add(db_transaction)
    db.commit()
    db.refresh(db_transaction)
    return db_transaction

@app.get("/transactions/")
def get_transactions(role: str, db: Session = Depends(get_db)):
    check_role(role, ["admin", "analyst", "viewer"])
    return db.query(Transaction).all()

@app.delete("/transactions/{id}")
def delete_transaction(id: int, role: str, db: Session = Depends(get_db)):
    check_role(role, ["admin"])
    t = db.query(Transaction).filter(Transaction.id == id).first()
    if not t:
        raise HTTPException(status_code=404, detail="Transaction not found")
    db.delete(t)
    db.commit()
    return {"message": "Deleted successfully"}

# ---------------- ANALYTICS ----------------
@app.get("/summary/")
def summary(role: str, db: Session = Depends(get_db)):
    check_role(role, ["admin", "analyst"])
    data = db.query(Transaction).all()
    income = sum(t.amount for t in data if t.type == "income")
    expense = sum(t.amount for t in data if t.type == "expense")
    return {
        "total_income": income,
        "total_expense": expense,
        "balance": income - expense
    }

@app.get("/category-summary/")
def category_summary(role: str, db: Session = Depends(get_db)):
    check_role(role, ["admin", "analyst"])
    data = db.query(Transaction).all()
    result = {}
    for t in data:
        result[t.category] = result.get(t.category, 0) + t.amount
    return result

@app.get("/monthly-summary/")
def monthly_summary(role: str, db: Session = Depends(get_db)):
    check_role(role, ["admin", "analyst"])
    data = db.query(Transaction).all()
    result = {}
    for t in data:
        month = t.date.strftime("%Y-%m")
        result[month] = result.get(month, 0) + t.amount
    return result

# ---------------- RUN SERVER ----------------
public_url = ngrok.connect(addr=8000, bind_tls=True)
print("👉 CLICK THIS LINK:", str(public_url) + "/docs")
def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run).start()

👉 CLICK THIS LINK: NgrokTunnel: "https://unmisinterpretable-lashandra-aghastly.ngrok-free.dev" -> "http://localhost:8000"/docs


INFO:     Started server process [9023]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
